# 08 — One-change sensitivity analyses

This notebook evaluates the five prespecified one-change sensitivity analyses for the locked final run. Each branch changes exactly one assumption of the primary 1,040-patient, 26-feature analysis:

1. add the three combined-state Part III unable-to-rate (`101`) flags;
2. replace combined-state Part III scores with separate ON/OFF scores;
3. use the broader 1,043-patient cohort;
4. use revision-compatible preprocessing;
5. use latest rather than maximum eligible constipation.

**Locked procedure:** every branch reuses Notebook 07's 20 outer splits, its 60 locally selected pipelines (candidate type, selector rule, engineering branch, model family, and hyperparameters), and its refit random seeds. Each branch refits the frozen pipeline on the outer-training patients and scores the untouched outer-test patients once. Fold-fitted steps such as imputation, encoding, scaling, PCA loadings, and selector decisions are refitted on the branch's training data, exactly as in the primary run.

**Selection boundary:** no tuning, pipeline choice, or model-family choice happens here. Sensitivity test results describe robustness only; they cannot redefine the primary pipeline or promote a branch.

## 1. Check the modeling environment

Confirm that the three external model packages used by the frozen winners are available. The CPU setting prevents a noisy joblib hardware-detection warning.

In [1]:
import importlib.util
import os

# Avoid noisy macOS physical-core detection in joblib while preserving the
# available logical-core limit for parallel estimators.
os.environ.setdefault(
    "LOKY_MAX_CPU_COUNT",
    str(max(1, (os.cpu_count() or 2) - 1)),
)

required_external_packages = ["xgboost", "lightgbm", "catboost"]
missing_packages = [
    package
    for package in required_external_packages
    if importlib.util.find_spec(package) is None
]
if missing_packages:
    raise ModuleNotFoundError(
        "Install the missing packages in this notebook environment, restart the "
        f"kernel, and rerun: {missing_packages}"
    )
print("External model packages are available.")

External model packages are available.


## 2. Import the analysis tools

Load the data, statistical, preprocessing, modeling, and evaluation tools used below.

In [2]:
from pathlib import Path
import hashlib
import json
import platform
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from sklearn import __version__ as sklearn_version
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    f1_score, recall_score, roc_auc_score,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier, __version__ as xgboost_version
from lightgbm import LGBMClassifier, __version__ as lightgbm_version
from catboost import CatBoostClassifier, __version__ as catboost_version

## 3. Locate the frozen inputs

Use the Notebook 02 sensitivity inputs, the Notebook 03 splits, the Notebook 05 engineering definitions, and the Notebook 07 winners and primary results. The Notebook 28b preflight manifest is loaded for traceability.

In [3]:
working_directory = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in [working_directory, *working_directory.parents]
        if (folder / "AGENTS.md").is_file()
        and (folder / "docs/research_protocol.md").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Launch this notebook from within the project directory.")

final_run_directory = project_root / "results/final_pipeline/06_final_fit_and_performance_summary"
aggregation_directory = final_run_directory / "02_patient_level_aggregation"
split_directory = final_run_directory / "03_setup_and_splits"
feature_screen_directory = final_run_directory / "05_feature_pipeline_screening"
primary_directory = final_run_directory / "07_tuning_and_outer_evaluation"
sensitivity_directory = project_root / "results/final_pipeline/07_sensitivity_analyses"
preflight_directory = sensitivity_directory / "preflight"
output_directory = sensitivity_directory / "08_sensitivity_analyses"
output_directory.mkdir(parents=True, exist_ok=True)

input_paths = {
    "primary_predictors": aggregation_directory / "primary_1040_26_predictors.csv",
    "primary_outcomes": aggregation_directory / "primary_1040_outcome_metadata.csv",
    "flag_predictors": aggregation_directory / "sensitivity_1040_part_iii_101_flags_predictors.csv",
    "on_off_predictors": aggregation_directory / "sensitivity_1040_part_iii_on_off_predictors.csv",
    "broad_predictors": aggregation_directory / "sensitivity_1043_26_predictors.csv",
    "broad_outcomes": aggregation_directory / "sensitivity_1043_outcome_metadata.csv",
    "latest_constipation_predictors": aggregation_directory / "sensitivity_1040_latest_constipation_predictors.csv",
    "feature_manifest": split_directory / "primary_feature_manifest.csv",
    "outer_splits": split_directory / "outer_split_assignments.csv",
    "engineering_manifest": feature_screen_directory / "engineering_configuration_manifest.csv",
    "tuning_grid": primary_directory / "tuning_grid_manifest.csv",
    "tuned_winners": primary_directory / "tuned_partition_winners.csv",
    "primary_selected_pipelines": primary_directory / "selected_pipelines.csv",
    "primary_predictions": primary_directory / "outer_predictions.csv",
    "primary_seed_metrics": primary_directory / "outer_seed_metrics.csv",
    "primary_tuning_validation": primary_directory / "focused_tuning_validation.csv",
    "primary_outer_validation": primary_directory / "outer_evaluation_validation.csv",
    "preflight_branches": preflight_directory / "sensitivity_branch_manifest.csv",
}
missing_inputs = [name for name, path in input_paths.items() if not path.is_file()]
assert not missing_inputs, f"Missing required inputs: {missing_inputs}"

print("Project root:", project_root)
print("Output directory:", output_directory.relative_to(project_root))

Project root: /Users/rafsan_temp/Library/CloudStorage/OneDrive-SeattleUniversity/SU Projects/pd-fall-risk
Output directory: results/final_pipeline/07_sensitivity_analyses/08_sensitivity_analyses


## 4. Load and validate the frozen inputs

Confirm the primary input, 20 paired splits, passed Notebook 07 validations, and one frozen winner per seed and target. The winners must match the configurations Notebook 07 refitted and must come from the frozen tuning grid.

In [4]:
def read_predictors(file_path):
    return pd.read_csv(
        file_path,
        dtype={"PATNO": "string"},
        low_memory=False,
    ).set_index("PATNO")


def read_outcomes(file_path):
    return pd.read_csv(
        file_path,
        dtype={"PATNO": "string"},
        usecols=["PATNO", "falls_class"],
    ).set_index("PATNO")["falls_class"].astype(int)


primary_predictors = read_predictors(input_paths["primary_predictors"])
flag_predictors = read_predictors(input_paths["flag_predictors"])
on_off_predictors = read_predictors(input_paths["on_off_predictors"])
broad_predictors = read_predictors(input_paths["broad_predictors"])
latest_constipation_predictors = read_predictors(input_paths["latest_constipation_predictors"])
primary_outcomes = read_outcomes(input_paths["primary_outcomes"])
broad_outcomes = read_outcomes(input_paths["broad_outcomes"])

feature_manifest = pd.read_csv(input_paths["feature_manifest"])
outer_assignments = pd.read_csv(input_paths["outer_splits"], dtype={"PATNO": "string"})
engineering_manifest = pd.read_csv(input_paths["engineering_manifest"])
tuning_grid = pd.read_csv(input_paths["tuning_grid"])
winners = pd.read_csv(input_paths["tuned_winners"])
primary_selected = pd.read_csv(input_paths["primary_selected_pipelines"])
primary_predictions = pd.read_csv(input_paths["primary_predictions"], dtype={"PATNO": "string"})
primary_seed_metrics = pd.read_csv(input_paths["primary_seed_metrics"])
primary_validations = pd.concat([
    pd.read_csv(input_paths["primary_tuning_validation"]),
    pd.read_csv(input_paths["primary_outer_validation"]),
])
preflight_branches = pd.read_csv(input_paths["preflight_branches"])

features = feature_manifest["feature"].tolist()
targets = ["direct", "stage_1", "stage_2"]
split_seeds = sorted(outer_assignments["split_seed"].unique())
winner_keys = [
    "split_seed", "target", "configuration_id", "model_family", "grid_id", "parameters",
]

assert primary_predictors.shape == (1040, 26)
assert primary_predictors.columns.tolist() == features
assert set(primary_predictors.index) == set(primary_outcomes.index)
assert split_seeds == list(range(20))
assert primary_validations["passed"].astype(str).str.lower().eq("true").all()
assert len(winners) == 60 and winners.groupby(["split_seed", "target"]).size().eq(1).all()
assert len(winners[winner_keys].merge(primary_selected[winner_keys])) == 60
assert len(winners.merge(tuning_grid, on=["model_family", "grid_id", "parameters"])) == 60
assert len(preflight_branches) == 5

print("Primary patients:", len(primary_predictors))
print("Frozen winners:", len(winners))
print("Passed Notebook 07 checks:", len(primary_validations))
display(winners.groupby(["target", "model_family"]).size().rename("frozen_winners").reset_index())

Primary patients: 1040
Frozen winners: 60
Passed Notebook 07 checks: 22


,target,model_family,frozen_winners
0,direct,catboost,7
1,direct,extra_trees,1
2,direct,linear_svc,1
3,direct,random_forest,9
4,direct,rbf_svc,2
5,stage_1,catboost,2
6,stage_1,extra_trees,6
7,stage_1,logistic,4
8,stage_1,random_forest,3
9,stage_1,rbf_svc,5


# Part A — Branch definitions

The next sections declare exactly what each sensitivity changes and verify that nothing else changes.

## 5. Declare the five one-change branches

The Notebook 28b preflight was written when the 1,043-patient cohort was primary. Decision D23 made the historical 1,040-patient cohort primary, so the cohort branch is reversed here: it **adds** the three patients without eligible longitudinal history. The other four branches keep the preflight definitions but are applied to the 1,040-patient primary cohort.

Three branch-specific rules were agreed before any sensitivity result existed:

- **ON/OFF (S02):** any frozen winner that uses an interaction built from combined-state Part III scores is handled as follows. Those scores do not exist in this branch, and there is no prespecified ON/OFF interaction. These winners therefore keep their model, hyperparameters, and dense input on all ON/OFF features but omit the interaction term. They are recorded as `not_constructible`.
- **1,043 cohort (S03):** every primary train/test assignment is unchanged. The three added patients join outer training in all 20 seeds, so the test patients stay identical and the comparison with the primary run is fully paired. The approved `NO_ELIGIBLE_LONGITUDINAL_HISTORY` indicator is added in this branch only.
- **Revision preprocessing (S04):** following the legacy revision code, missing Part IV totals become zero, nominal gaps use the training mode, dopamine history uses the training mode plus `DOPTHERST_MISSING`, and no form-missing indicators are added. Dense models use training medians for numeric gaps. Native-missing boosting models keep numeric `NaN`, as they did in the revision pipeline.

A primary reproduction branch, `P00`, reruns Notebook 07's outer refits with this notebook's code. The sensitivities run only after it reproduces the saved primary predictions exactly.

In [5]:
primary_train_ids = {}
primary_test_ids = {}
for split_seed in split_seeds:
    seed_rows = outer_assignments.loc[outer_assignments["split_seed"].eq(split_seed)]
    primary_train_ids[split_seed] = sorted(seed_rows.loc[seed_rows["role"].eq("train"), "PATNO"])
    primary_test_ids[split_seed] = sorted(seed_rows.loc[seed_rows["role"].eq("test"), "PATNO"])

added_cohort_ids = sorted(set(broad_predictors.index) - set(primary_predictors.index))

branch_specs = {
    "P00_PRIMARY_REPRODUCTION": {
        "changed_assumption": "None; reproduces Notebook 07 before any sensitivity",
        "predictors": primary_predictors,
        "outcomes": primary_outcomes,
        "preprocessing": "current",
        "history_indicator": False,
        "extra_training_ids": [],
        "branch_rule": "Must reproduce the saved Notebook 07 predictions",
    },
    "S01_101_FLAGS": {
        "changed_assumption": "Add three combined-state Part III ever-101 flags",
        "predictors": flag_predictors,
        "outcomes": primary_outcomes,
        "preprocessing": "current",
        "history_indicator": False,
        "extra_training_ids": [],
        "branch_rule": "Each flag is its own selection group and is tested as categorical",
    },
    "S02_ON_OFF": {
        "changed_assumption": "Replace four combined-state Part III maxima with eight ON/OFF maxima",
        "predictors": on_off_predictors,
        "outcomes": primary_outcomes,
        "preprocessing": "current",
        "history_indicator": False,
        "extra_training_ids": [],
        "branch_rule": (
            "Add ON/OFF state-availability indicators grouped with their scores; "
            "combined-state interactions are not constructible and are omitted"
        ),
    },
    "S03_COHORT_1043": {
        "changed_assumption": "Add the three patients without eligible longitudinal history",
        "predictors": broad_predictors,
        "outcomes": broad_outcomes,
        "preprocessing": "current",
        "history_indicator": True,
        "extra_training_ids": added_cohort_ids,
        "branch_rule": (
            "Primary splits unchanged; added patients join outer training in every "
            "seed; add NO_ELIGIBLE_LONGITUDINAL_HISTORY"
        ),
    },
    "S04_REVISION_PREPROCESSING": {
        "changed_assumption": "Use revision-compatible missing-data rules",
        "predictors": primary_predictors,
        "outcomes": primary_outcomes,
        "preprocessing": "revision",
        "history_indicator": False,
        "extra_training_ids": [],
        "branch_rule": (
            "Blanket Part IV zero; nominal training mode; dopamine mode plus "
            "DOPTHERST_MISSING; no form indicators; native models keep numeric NaN"
        ),
    },
    "S05_CONSTIPATION_LATEST": {
        "changed_assumption": "Use latest rather than maximum eligible NP1CNST",
        "predictors": latest_constipation_predictors,
        "outcomes": primary_outcomes,
        "preprocessing": "current",
        "history_indicator": False,
        "extra_training_ids": [],
        "branch_rule": "Only the NP1CNST values change",
    },
}
SENSITIVITY_IDS = [name for name in branch_specs if name.startswith("S")]

branch_manifest = pd.DataFrame([
    {
        "sensitivity_id": name,
        "changed_assumption": spec["changed_assumption"],
        "branch_rule": spec["branch_rule"],
        "patients": len(spec["predictors"]),
        "predictor_columns": spec["predictors"].shape[1],
        "preprocessing": spec["preprocessing"],
        "added_training_patients": len(spec["extra_training_ids"]),
        "locked_procedure": "Notebook 07 splits, winners, hyperparameters, and refit seeds",
        "selection_role": "reporting only; cannot redefine the primary pipeline",
    }
    for name, spec in branch_specs.items()
])
display(branch_manifest[[
    "sensitivity_id", "changed_assumption", "patients", "predictor_columns", "preprocessing",
]])

,sensitivity_id,changed_assumption,patients,predictor_columns,preprocessing
0,P00_PRIMARY_REPRODUCTION,None; reproduces Notebook 07 before any sensit...,1040,26,current
1,S01_101_FLAGS,Add three combined-state Part III ever-101 flags,1040,29,current
2,S02_ON_OFF,Replace four combined-state Part III maxima wi...,1040,30,current
3,S03_COHORT_1043,Add the three patients without eligible longit...,1043,26,current
4,S04_REVISION_PREPROCESSING,Use revision-compatible missing-data rules,1040,26,revision
5,S05_CONSTIPATION_LATEST,Use latest rather than maximum eligible NP1CNST,1040,26,current


## 6. Verify that each branch changes only one assumption

Compare every sensitivity input with the primary input. Shared columns and shared patients must be identical, so any performance difference can be attributed to the declared change. This check uses predictors only.

In [6]:
PART_III_COMBINED = [
    "NP3GAIT_COMBINED_MAX", "NP3PSTBL_COMBINED_MAX", "NHY_COMBINED_MAX", "NP3TOT_COMBINED_MAX",
]
PART_III_ON = ["NP3TOT_ON_MAX", "NP3GAIT_ON_MAX", "NP3PSTBL_ON_MAX", "NHY_ON_MAX"]
PART_III_OFF = ["NP3TOT_OFF_MAX", "NP3GAIT_OFF_MAX", "NP3PSTBL_OFF_MAX", "NHY_OFF_MAX"]
PART_III_101_FLAGS = ["NP3GAIT_WAS_101_EVER", "NP3PSTBL_WAS_101_EVER", "NHY_WAS_101_EVER"]
BASELINE_FEATURES = {
    "Years_since_PD_diagnosis", "Age", "DXPOSINS", "DXRIGID",
    "DXTREMOR", "DXBRADY", "DOMSIDE", "ANYFAMPD",
}
LONGITUDINAL_FEATURES = [feature for feature in features if feature not in BASELINE_FEATURES]


def shared_values_identical(sensitivity, columns, patients):
    # Added patients' gaps can turn an integer column into floats without changing values.
    for column in columns:
        try:
            pd.testing.assert_series_equal(
                sensitivity.loc[patients, column],
                primary_predictors.loc[patients, column],
                check_dtype=False,
                check_exact=True,
            )
        except AssertionError:
            return False
    return True


primary_ids = primary_predictors.index
added_rows = broad_predictors.loc[added_cohort_ids]
constipation_changed = latest_constipation_predictors.loc[primary_ids, "NP1CNST"].ne(
    primary_predictors["NP1CNST"]
).sum()
shared_on_off = [column for column in features if column not in PART_III_COMBINED]
shared_constipation = [column for column in features if column != "NP1CNST"]

one_change_audit = pd.DataFrame([
    {
        "sensitivity_id": "S01_101_FLAGS",
        "check": "adds only the three flags; flags complete and binary",
        "passed": flag_predictors.columns.tolist() == [*features, *PART_III_101_FLAGS]
        and set(flag_predictors.index) == set(primary_ids)
        and shared_values_identical(flag_predictors, features, primary_ids)
        and flag_predictors[PART_III_101_FLAGS].isin([0, 1]).all().all(),
        "detail": "flagged patients: " + ", ".join(
            f"{column}={int(flag_predictors[column].sum())}" for column in PART_III_101_FLAGS
        ),
    },
    {
        "sensitivity_id": "S02_ON_OFF",
        "check": "replaces only the four combined-state scores",
        "passed": set(on_off_predictors.columns)
        == (set(features) - set(PART_III_COMBINED)) | set(PART_III_ON) | set(PART_III_OFF)
        and set(on_off_predictors.index) == set(primary_ids)
        and shared_values_identical(on_off_predictors, shared_on_off, primary_ids),
        "detail": (
            f"ON available {int(on_off_predictors[PART_III_ON].notna().any(axis=1).sum())}; "
            f"OFF available {int(on_off_predictors[PART_III_OFF].notna().any(axis=1).sum())}"
        ),
    },
    {
        "sensitivity_id": "S03_COHORT_1043",
        "check": "adds three patients without longitudinal history; shared rows identical",
        "passed": broad_predictors.columns.tolist() == features
        and len(added_cohort_ids) == 3
        and set(primary_ids) < set(broad_predictors.index)
        and shared_values_identical(broad_predictors, features, primary_ids)
        and broad_outcomes.loc[primary_ids].equals(primary_outcomes.loc[primary_ids])
        and added_rows[LONGITUDINAL_FEATURES].isna().all().all()
        and broad_predictors[LONGITUDINAL_FEATURES].isna().all(axis=1).sum() == 3,
        "detail": "added patient classes: " + str(
            broad_outcomes.loc[added_cohort_ids].value_counts().sort_index().to_dict()
        ),
    },
    {
        "sensitivity_id": "S04_REVISION_PREPROCESSING",
        "check": "uses the unchanged primary input",
        "passed": branch_specs["S04_REVISION_PREPROCESSING"]["predictors"] is primary_predictors,
        "detail": "only the fold-fitted missing-data rules change",
    },
    {
        "sensitivity_id": "S05_CONSTIPATION_LATEST",
        "check": "changes only NP1CNST",
        "passed": latest_constipation_predictors.columns.tolist() == features
        and set(latest_constipation_predictors.index) == set(primary_ids)
        and shared_values_identical(latest_constipation_predictors, shared_constipation, primary_ids)
        and latest_constipation_predictors["NP1CNST"].notna().all(),
        "detail": f"patients with a different NP1CNST value: {int(constipation_changed)}",
    },
])
display(one_change_audit)
assert one_change_audit["passed"].all(), one_change_audit.loc[~one_change_audit["passed"]]

,sensitivity_id,check,passed,detail
0,S01_101_FLAGS,adds only the three flags; flags complete and ...,True,"flagged patients: NP3GAIT_WAS_101_EVER=53, NP3..."
1,S02_ON_OFF,replaces only the four combined-state scores,True,ON available 795; OFF available 729
2,S03_COHORT_1043,adds three patients without longitudinal histo...,True,added patient classes: {0: 3}
3,S04_REVISION_PREPROCESSING,uses the unchanged primary input,True,only the fold-fitted missing-data rules change
4,S05_CONSTIPATION_LATEST,changes only NP1CNST,True,patients with a different NP1CNST value: 381


# Part B — Locked pipeline definitions

The next sections recreate Notebook 07's fitted pipeline. The only extensions are the branch-specific columns and the revision preprocessing option; with the primary input, the code follows the Notebook 07 path exactly.

## 7. Recreate the locked model definitions

Build each frozen winner with the same estimator settings and backend assignment as Notebook 07.

In [7]:
MODEL_BACKEND = {
    "logistic": "scaled_dense",
    "linear_svc": "scaled_dense",
    "rbf_svc": "scaled_dense",
    "random_forest": "unscaled_dense",
    "extra_trees": "unscaled_dense",
    "hist_gradient_boosting": "native",
    "xgboost": "native",
    "lightgbm": "native",
    "catboost": "native",
}


def build_model(family, target_name, random_seed, parameters, probability=False):
    parameters = dict(parameters)
    if family == "logistic":
        return LogisticRegression(
            max_iter=5000,
            random_state=random_seed,
            **parameters,
        ), None
    if family == "linear_svc":
        return LinearSVC(
            dual="auto",
            max_iter=10000,
            random_state=random_seed,
            **parameters,
        ), None
    if family == "rbf_svc":
        return SVC(
            cache_size=1000,
            probability=probability,
            random_state=random_seed,
            **parameters,
        ), None
    if family == "random_forest":
        return RandomForestClassifier(
            random_state=random_seed,
            n_jobs=-1,
            **parameters,
        ), None
    if family == "extra_trees":
        return ExtraTreesClassifier(
            random_state=random_seed,
            n_jobs=-1,
            **parameters,
        ), None
    if family == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(
            random_state=random_seed,
            **parameters,
        ), None
    if family == "xgboost":
        weight_mode = parameters.pop("sample_weight")
        class_count = 3 if target_name == "direct" else 2
        parameters.update({
            "objective": "multi:softprob" if class_count == 3 else "binary:logistic",
            "eval_metric": "mlogloss" if class_count == 3 else "logloss",
            "random_state": random_seed,
            "n_jobs": -1,
            "verbosity": 0,
        })
        if class_count == 3:
            parameters["num_class"] = 3
        return XGBClassifier(**parameters), weight_mode
    if family == "lightgbm":
        return LGBMClassifier(
            random_state=random_seed,
            n_jobs=-1,
            verbosity=-1,
            **parameters,
        ), None

    loss = "MultiClass" if target_name == "direct" else "Logloss"
    return CatBoostClassifier(
        random_seed=random_seed,
        loss_function=loss,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
        **parameters,
    ), None

## 8. Define feature families and clinical preprocessing

Extend the Notebook 07 feature families with the sensitivity columns. The approved rules in `docs/final_run/final_imputation_rules.md` apply:

- Each ON/OFF score block is grouped with its state-availability indicator.
- `DOPTHERST_MISSING` is grouped with dopamine history.
- The `101` flags and the longitudinal-history indicator are independent groups.

Every derived indicator is created from the original missingness pattern before any value is filled.

In [8]:
NOMINAL_FEATURES = {
    "DXPOSINS", "DXRIGID", "DOPTHERST", "FEATPOSHYP",
    "ANYFAMPD", "DXTREMOR", "DXBRADY", "DOMSIDE",
}
ORDINAL_FEATURES = {
    "FRZGT12M", "SCAU14", "SCAU16", "NP1SLPD", "NP1URIN",
    "NP3GAIT_COMBINED_MAX", "NP3PSTBL_COMBINED_MAX", "NHY_COMBINED_MAX",
    "NP1CNST",
    "NP3GAIT_ON_MAX", "NP3PSTBL_ON_MAX", "NHY_ON_MAX",
    "NP3GAIT_OFF_MAX", "NP3PSTBL_OFF_MAX", "NHY_OFF_MAX",
}
BINARY_INDICATORS = {
    "FOG_FORM_MISSING", "NQ_FORM_MISSING", "PART_IV_FORM_MISSING",
    "ON_STATE_AVAILABLE", "OFF_STATE_AVAILABLE",
    "NO_ELIGIBLE_LONGITUDINAL_HISTORY", "DOPTHERST_MISSING",
    *PART_III_101_FLAGS,
}
SENSITIVITY_COLUMNS = {
    "S01_101_FLAGS": PART_III_101_FLAGS,
    "S02_ON_OFF": [*PART_III_ON, *PART_III_OFF, "ON_STATE_AVAILABLE", "OFF_STATE_AVAILABLE"],
    "S03_COHORT_1043": ["NO_ELIGIBLE_LONGITUDINAL_HISTORY"],
    "S04_REVISION_PREPROCESSING": ["DOPTHERST_MISSING"],
    "S05_CONSTIPATION_LATEST": ["NP1CNST"],
}


def representation_group(source):
    if source in {"FRZGT12M", "FOG_FORM_MISSING"}:
        return "GROUP_FREEZING_FORM"
    if source in {"NQ_GAUSSIAN_REVISION", "NQ_FORM_MISSING"}:
        return "GROUP_NEUROQOL_FORM"
    if source in {"NP4TOT", "PART_IV_FORM_MISSING"}:
        return "GROUP_PART_IV_FORM"
    if source in {*PART_III_ON, "ON_STATE_AVAILABLE"}:
        return "GROUP_PART_III_ON_STATE"
    if source in {*PART_III_OFF, "OFF_STATE_AVAILABLE"}:
        return "GROUP_PART_III_OFF_STATE"
    if source == "DOPTHERST_MISSING":
        return "DOPTHERST"
    return source


def clinical_frame(frame, preprocessing, history_indicator):
    raw = frame.copy()
    indicators = pd.DataFrame(index=raw.index)
    if preprocessing == "current":
        indicators["FOG_FORM_MISSING"] = raw["FRZGT12M"].isna().astype(int)
        indicators["NQ_FORM_MISSING"] = raw["NQ_GAUSSIAN_REVISION"].isna().astype(int)
        indicators["PART_IV_FORM_MISSING"] = raw["NP4TOT"].isna().astype(int)
        structural_zero = raw["NP4TOT"].isna() & raw["DOPTHERST"].eq("No")
        raw.loc[structural_zero, "NP4TOT"] = 0.0
    elif preprocessing == "revision":
        indicators["DOPTHERST_MISSING"] = raw["DOPTHERST"].isna().astype(int)
        raw["NP4TOT"] = raw["NP4TOT"].fillna(0.0)
    else:
        raise ValueError(f"Unknown preprocessing branch: {preprocessing}")

    if set(PART_III_ON).issubset(raw.columns):
        indicators["ON_STATE_AVAILABLE"] = raw[PART_III_ON].notna().any(axis=1).astype(int)
    if set(PART_III_OFF).issubset(raw.columns):
        indicators["OFF_STATE_AVAILABLE"] = raw[PART_III_OFF].notna().any(axis=1).astype(int)
    if history_indicator:
        indicators["NO_ELIGIBLE_LONGITUDINAL_HISTORY"] = (
            raw[LONGITUDINAL_FEATURES].isna().all(axis=1).astype(int)
        )
    return pd.concat([raw, indicators], axis=1)

## 9. Define the corrected statistical selector

This is Notebook 07's corrected FDR selector. It tests observed training values only and groups encoded columns with their clinical source. Binary flags and indicators use the categorical test.

In [9]:
def bh_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan)
    valid = np.flatnonzero(np.isfinite(p_values))
    if not len(valid):
        return adjusted
    order = valid[np.argsort(p_values[valid])]
    ranked = p_values[order] * len(valid) / np.arange(1, len(valid) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted[order] = np.minimum(ranked, 1.0)
    return adjusted


def numeric_p_value(values, target, ordinal):
    observed = pd.DataFrame({"value": values, "target": target}).dropna()
    groups = [
        observed.loc[observed["target"].eq(label), "value"].astype(float).to_numpy()
        for label in sorted(observed["target"].unique())
    ]
    if len(groups) < 2 or min(len(group) for group in groups) < 2:
        return np.nan
    if np.unique(np.concatenate(groups)).size < 2:
        return 1.0

    if ordinal:
        test = (
            stats.mannwhitneyu(*groups, alternative="two-sided")
            if len(groups) == 2
            else stats.kruskal(*groups)
        )
        return float(test.pvalue)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        normality = [
            stats.normaltest(group).pvalue
            if len(group) >= 8 and np.unique(group).size >= 3
            else 0.0
            for group in groups
        ]
        variance_p = stats.levene(*groups, center="median").pvalue
    parametric = (
        all(np.isfinite(normality))
        and min(normality) >= 0.05
        and np.isfinite(variance_p)
        and variance_p >= 0.05
    )
    if len(groups) == 2:
        test = (
            stats.ttest_ind(*groups, equal_var=True)
            if parametric
            else stats.mannwhitneyu(*groups, alternative="two-sided")
        )
    else:
        test = stats.f_oneway(*groups) if parametric else stats.kruskal(*groups)
    return float(test.pvalue)


def categorical_p_value(values, target, random_seed):
    categories = values.astype("string").fillna("Missing")
    table = pd.crosstab(categories, target)
    table = table.loc[table.sum(axis=1).gt(0)]
    if table.shape[0] < 2 or table.shape[1] < 2:
        return 1.0
    asymptotic = stats.chi2_contingency(table, correction=False)
    sparse = (
        asymptotic.expected_freq.min() < 1
        or (asymptotic.expected_freq < 5).mean() > 0.20
    )
    if not sparse:
        return float(asymptotic.pvalue)
    method = stats.PermutationMethod(
        n_resamples=999,
        rng=np.random.default_rng(random_seed),
    )
    return float(
        stats.chi2_contingency(
            table,
            correction=False,
            method=method,
        ).pvalue
    )


def select_groups_fdr(raw, target, random_seed):
    p_values = []
    for position, column in enumerate(raw.columns):
        if column in NOMINAL_FEATURES or column in BINARY_INDICATORS:
            p_value = categorical_p_value(
                raw[column],
                target,
                random_seed + position,
            )
        else:
            p_value = numeric_p_value(
                raw[column],
                target,
                ordinal=column in ORDINAL_FEATURES,
            )
        p_values.append(p_value)

    q_values = bh_adjust(p_values)
    selected_sources = [
        column
        for column, q_value in zip(raw.columns, q_values)
        if np.isfinite(q_value) and q_value <= 0.05
    ]
    if not selected_sources:
        finite = np.flatnonzero(np.isfinite(q_values))
        fallback = finite[np.argmin(q_values[finite])] if len(finite) else 0
        selected_sources = [raw.columns[fallback]]
    return {representation_group(source) for source in selected_sources}

## 10. Define the fold-fitted candidate transformer

This is Notebook 07's transformer with two additions. `preprocessing="revision"` applies the revision missing-data rules. An interaction whose source columns are absent is recorded as `not_constructible` instead of failing; PCA blocks must always be constructible. The longer cell is kept intact because these steps share fitted state.

In [10]:
engineering_definitions = {
    row.branch: {
        "kind": row.branch_kind,
        "sources": row.source_features.split(" | "),
        "threshold": None if pd.isna(row.variance_threshold) else float(row.variance_threshold),
    }
    for row in engineering_manifest.itertuples(index=False)
}


class CandidateTransformer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        candidate_kind,
        selector,
        selector_parameter,
        branch=None,
        backend="scaled_dense",
        preprocessing="current",
        history_indicator=False,
        random_state=42,
    ):
        self.candidate_kind = candidate_kind
        self.selector = selector
        self.selector_parameter = selector_parameter
        self.branch = branch
        self.backend = backend
        self.preprocessing = preprocessing
        self.history_indicator = history_indicator
        self.random_state = random_state

    def _nominal_frame(self, raw):
        nominal = raw[self.nominal_columns_].astype("string")
        if self.preprocessing == "revision":
            nominal = nominal.fillna(self.nominal_modes_)
        return nominal.fillna("Missing").astype(str)

    def _fit_preprocessing(self, raw):
        self.nominal_columns_ = [column for column in raw if column in NOMINAL_FEATURES]
        self.numeric_columns_ = [column for column in raw if column not in self.nominal_columns_]

        self.nominal_modes_ = {}
        if self.preprocessing == "revision":
            for column in self.nominal_columns_:
                mode = raw[column].mode(dropna=True)
                if mode.empty:
                    raise ValueError(f"{column} has no observed training category.")
                self.nominal_modes_[column] = str(mode.iloc[0])

        freezing_mode = raw["FRZGT12M"].mode(dropna=True)
        if freezing_mode.empty:
            raise ValueError("FRZGT12M has no observed training value.")
        self.freezing_fill_ = float(freezing_mode.iloc[0])

        median_columns = [column for column in self.numeric_columns_ if column != "FRZGT12M"]
        self.medians_ = raw[median_columns].median()
        if self.medians_.isna().any():
            missing = self.medians_[self.medians_.isna()].index.tolist()
            raise ValueError(f"No training value available for: {missing}")

        dense_numeric = raw[self.numeric_columns_].copy()
        dense_numeric["FRZGT12M"] = dense_numeric["FRZGT12M"].fillna(
            self.freezing_fill_
        )
        dense_numeric[median_columns] = dense_numeric[median_columns].fillna(
            self.medians_
        )
        self.scaler_ = StandardScaler().fit(dense_numeric)

        self.encoder_ = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ).fit(self._nominal_frame(raw))
        self.encoded_nominal_columns_ = self.encoder_.get_feature_names_out(
            self.nominal_columns_
        ).tolist()

        source_by_column = {column: column for column in self.numeric_columns_}
        encoded_sources = [
            source
            for source, categories in zip(
                self.nominal_columns_,
                self.encoder_.categories_,
            )
            for _ in categories
        ]
        source_by_column.update(
            dict(zip(self.encoded_nominal_columns_, encoded_sources))
        )
        self.group_by_column_ = {
            column: representation_group(source)
            for column, source in source_by_column.items()
        }

    def _matrices(self, raw):
        dense_numeric = raw[self.numeric_columns_].copy()
        dense_numeric["FRZGT12M"] = dense_numeric["FRZGT12M"].fillna(
            self.freezing_fill_
        )
        median_columns = [column for column in self.numeric_columns_ if column != "FRZGT12M"]
        dense_numeric[median_columns] = dense_numeric[median_columns].fillna(
            self.medians_
        )
        scaled_numeric = pd.DataFrame(
            self.scaler_.transform(dense_numeric),
            index=raw.index,
            columns=self.numeric_columns_,
        )
        encoded = pd.DataFrame(
            self.encoder_.transform(self._nominal_frame(raw)),
            index=raw.index,
            columns=self.encoded_nominal_columns_,
        )
        native_numeric = raw[self.numeric_columns_].apply(
            pd.to_numeric,
            errors="coerce",
        )
        return {
            "scaled_dense": pd.concat([scaled_numeric, encoded], axis=1),
            "unscaled_dense": pd.concat([dense_numeric, encoded], axis=1),
            "native": pd.concat([native_numeric, encoded], axis=1),
        }

    def _apply_engineering(self, matrix, fit):
        if self.candidate_kind != "engineered_representation":
            return matrix
        definition = engineering_definitions[self.branch]
        sources = definition["sources"]
        if fit:
            constructible = set(sources).issubset(matrix.columns)
            if not constructible and definition["kind"] != "interaction":
                raise ValueError(f"PCA sources are unavailable for {self.branch}.")
            self.engineering_status_ = "constructed" if constructible else "not_constructible"
        if self.engineering_status_ == "not_constructible":
            return matrix
        matrix = matrix.copy()

        if definition["kind"] == "interaction":
            matrix[self.branch] = (
                matrix[sources[0]].to_numpy()
                * matrix[sources[1]].to_numpy()
            )
            return matrix

        if fit:
            self.pca_ = PCA(
                n_components=definition["threshold"],
                svd_solver="full",
            ).fit(matrix[sources])
        components = self.pca_.transform(matrix[sources])
        matrix = matrix.drop(columns=sources)
        names = [
            f"{self.branch}_PC{index + 1}"
            for index in range(components.shape[1])
        ]
        matrix[names] = components
        return matrix

    def _select_columns(self, raw, dense, target):
        if self.selector == "none":
            self.selected_groups_ = set(self.group_by_column_.values())
            return dense.columns.tolist()

        if self.selector == "corrected_fdr":
            selected_groups = select_groups_fdr(raw, target, self.random_state)
        elif self.selector == "l1":
            C_value = float(self.selector_parameter.split("=")[1])
            base_estimator = LogisticRegression(
                solver="liblinear",
                l1_ratio=1.0,
                C=C_value,
                class_weight="balanced",
                max_iter=5000,
                random_state=self.random_state,
            )
            selector = OneVsRestClassifier(base_estimator, n_jobs=-1)
            selector.fit(dense, target)
            importance = np.max(
                np.vstack([
                    np.abs(estimator.coef_).reshape(-1)
                    for estimator in selector.estimators_
                ]),
                axis=0,
            )
            selected_groups = {
                self.group_by_column_[column]
                for column, value in zip(dense.columns, importance)
                if value > 1e-10
            }
            if not selected_groups:
                strongest = dense.columns[int(np.argmax(importance))]
                selected_groups = {self.group_by_column_[strongest]}
        else:
            multiplier = 1.25 if "1.25" in self.selector_parameter else 1.0
            selector = ExtraTreesClassifier(
                n_estimators=120,
                max_depth=6,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=self.random_state,
                n_jobs=-1,
            )
            selector.fit(dense, target)
            group_importance = {}
            for column, value in zip(dense.columns, selector.feature_importances_):
                group = self.group_by_column_[column]
                group_importance[group] = group_importance.get(group, 0.0) + float(value)
            threshold = np.median(list(group_importance.values())) * multiplier
            selected_groups = {
                group
                for group, value in group_importance.items()
                if value >= threshold
            }
            if not selected_groups:
                selected_groups = {max(group_importance, key=group_importance.get)}

        self.selected_groups_ = selected_groups
        return [
            column
            for column in dense
            if self.group_by_column_[column] in selected_groups
        ]

    def fit(self, X, y=None):
        raw = clinical_frame(X, self.preprocessing, self.history_indicator)
        self._fit_preprocessing(raw)
        matrices = self._matrices(raw)
        self.engineering_status_ = "not_applicable"
        dense = self._apply_engineering(matrices["scaled_dense"], fit=True)

        if self.candidate_kind == "engineered_representation":
            self.force_dense_ = True
            self.selected_columns_ = dense.columns.tolist()
            self.selected_groups_ = set(self.group_by_column_.values())
            if self.engineering_status_ == "constructed":
                self.selected_groups_.add(self.branch)
        else:
            self.force_dense_ = False
            self.selected_columns_ = self._select_columns(raw, dense, y)
        if not self.selected_columns_:
            raise RuntimeError("Candidate transformer selected no columns.")
        return self

    def transform(self, X):
        raw = clinical_frame(X, self.preprocessing, self.history_indicator)
        matrices = self._matrices(raw)
        if self.force_dense_:
            output = self._apply_engineering(
                matrices["scaled_dense"],
                fit=False,
            )
        else:
            output = matrices[self.backend]
        return output[self.selected_columns_]

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.selected_columns_, dtype=object)

## 11. Preview each branch's fold-fitted model input

Before evaluation, fit only the preprocessing on seed 0's outer-training patients for every branch and transform its test patients. No outcome is used and no model is fitted. Dense inputs must be complete; native inputs may retain numeric `NaN`.

In [11]:
preview_rows = []
for name, spec in branch_specs.items():
    training_ids = sorted([*primary_train_ids[0], *spec["extra_training_ids"]])
    test_ids = primary_test_ids[0]
    for backend in ["scaled_dense", "native"]:
        preview = CandidateTransformer(
            candidate_kind="selector_configuration",
            selector="none",
            selector_parameter="all",
            backend=backend,
            preprocessing=spec["preprocessing"],
            history_indicator=spec["history_indicator"],
        ).fit(spec["predictors"].loc[training_ids])
        transformed = preview.transform(spec["predictors"].loc[test_ids])
        derived = [
            column for column in preview.numeric_columns_
            if column not in spec["predictors"].columns
        ]
        preview_rows.append({
            "sensitivity_id": name,
            "backend": backend,
            "model_columns": transformed.shape[1],
            "selection_groups": len(set(preview.group_by_column_.values())),
            "test_missing_cells": int(transformed.isna().sum().sum()),
            "derived_indicators": " | ".join(derived),
            "revision_dopamine_fill": preview.nominal_modes_.get("DOPTHERST"),
        })
branch_preview = pd.DataFrame(preview_rows)
display(branch_preview)
assert branch_preview.loc[
    branch_preview["backend"].eq("scaled_dense"),
    "test_missing_cells",
].eq(0).all()

,sensitivity_id,backend,model_columns,selection_groups,test_missing_cells,derived_indicators,revision_dopamine_fill
0,P00_PRIMARY_REPRODUCTION,scaled_dense,47,26,0,FOG_FORM_MISSING | NQ_FORM_MISSING | PART_IV_F...,None
1,P00_PRIMARY_REPRODUCTION,native,47,26,56,FOG_FORM_MISSING | NQ_FORM_MISSING | PART_IV_F...,None
2,S01_101_FLAGS,scaled_dense,50,29,0,FOG_FORM_MISSING | NQ_FORM_MISSING | PART_IV_F...,None
3,S01_101_FLAGS,native,50,29,56,FOG_FORM_MISSING | NQ_FORM_MISSING | PART_IV_F...,None
4,S02_ON_OFF,scaled_dense,53,24,0,FOG_FORM_MISSING | NQ_FORM_MISSING | PART_IV_F...,None
5,S02_ON_OFF,native,53,24,657,FOG_FORM_MISSING | NQ_FORM_MISSING | PART_IV_F...,None
6,S03_COHORT_1043,scaled_dense,49,27,0,FOG_FORM_MISSING | NQ_FORM_MISSING | PART_IV_F...,None
7,S03_COHORT_1043,native,49,27,56,FOG_FORM_MISSING | NQ_FORM_MISSING | PART_IV_F...,None
8,S04_REVISION_PREPROCESSING,scaled_dense,42,26,0,DOPTHERST_MISSING,Yes
9,S04_REVISION_PREPROCESSING,native,42,26,38,DOPTHERST_MISSING,Yes


## 12. Build target outcomes, metrics, and warning handling

Define target-specific labels, the Notebook 07 probability conversion, and the fit wrapper that stops with one concise message if a model emits a warning.

In [12]:
def fit_checked(estimator, X, y, context, **fit_parameters):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        estimator.fit(X, y, **fit_parameters)

    if caught:
        unique_messages = list(dict.fromkeys(str(item.message) for item in caught))
        convergence = any(
            issubclass(item.category, ConvergenceWarning)
            for item in caught
        )
        warning_type = "convergence warning" if convergence else "model warning"
        raise RuntimeError(
            f"{context} emitted a {warning_type}: {unique_messages[0]}"
        )
    return estimator


def aligned_score_matrix(pipeline, frame, labels):
    model = pipeline.named_steps["model"]
    if hasattr(model, "predict_proba"):
        raw = np.asarray(pipeline.predict_proba(frame), dtype=float)
        score_source = "predict_proba"
    else:
        decision = np.asarray(pipeline.decision_function(frame), dtype=float)
        if decision.ndim == 1:
            positive = 1.0 / (1.0 + np.exp(-np.clip(decision, -700, 700)))
            raw = np.column_stack([1.0 - positive, positive])
        else:
            shifted = decision - decision.max(axis=1, keepdims=True)
            exponentiated = np.exp(shifted)
            raw = exponentiated / exponentiated.sum(axis=1, keepdims=True)
        score_source = "normalized_decision_score"

    aligned = np.zeros((len(raw), len(labels)), dtype=float)
    classes = np.asarray(model.classes_).astype(int)
    for source_column, label in enumerate(classes):
        aligned[:, labels.index(int(label))] = raw[:, source_column]
    assert np.isfinite(aligned).all()
    assert np.allclose(aligned.sum(axis=1), 1.0)
    return aligned, score_source


def multiclass_metrics(truth, predictions, scores):
    recalls = recall_score(
        truth,
        predictions,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )
    return {
        "macro_f1": f1_score(
            truth,
            predictions,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            truth,
            predictions,
            labels=[0, 1, 2],
            average="weighted",
            zero_division=0,
        ),
        "accuracy": accuracy_score(truth, predictions),
        "balanced_accuracy": balanced_accuracy_score(truth, predictions),
        "macro_auc_ovr": roc_auc_score(
            truth,
            scores,
            labels=[0, 1, 2],
            multi_class="ovr",
            average="macro",
        ),
        "recall_no_fall": recalls[0],
        "recall_rare_fall": recalls[1],
        "recall_recurrent_fall": recalls[2],
    }

## 13. Define the locked refit

Refit one frozen winner on a branch's outer-training patients and score the outer-test patients once. Stage 2 trains only on true fallers. The refit seed is Notebook 07's `500000 + split_seed`, so every branch uses the same randomness as the primary run.

In [13]:
def fit_locked_pipeline(sensitivity_id, winner, training_ids, test_ids, random_seed):
    spec = branch_specs[sensitivity_id]
    predictors = spec["predictors"]
    target_name = winner.target
    y_original = spec["outcomes"].loc[training_ids]
    model_training_ids = list(training_ids)
    if target_name == "direct":
        y_train = y_original
        labels = [0, 1, 2]
    elif target_name == "stage_1":
        y_train = y_original.gt(0).astype(int)
        labels = [0, 1]
    else:
        y_train = y_original[y_original.gt(0)].eq(2).astype(int)
        model_training_ids = y_train.index.tolist()
        labels = [0, 1]

    branch = None if pd.isna(winner.branch) else winner.branch
    transformer_instance = CandidateTransformer(
        candidate_kind=winner.candidate_kind,
        selector=winner.selector,
        selector_parameter=winner.selector_parameter,
        branch=branch,
        backend=MODEL_BACKEND[winner.model_family],
        preprocessing=spec["preprocessing"],
        history_indicator=spec["history_indicator"],
        random_state=random_seed,
    )
    estimator, weight_mode = build_model(
        winner.model_family,
        target_name,
        random_seed,
        json.loads(winner.parameters),
        probability=winner.model_family == "rbf_svc",
    )
    pipeline = Pipeline([
        ("candidate", transformer_instance),
        ("model", estimator),
    ])
    fit_parameters = {}
    if weight_mode == "balanced":
        fit_parameters["model__sample_weight"] = compute_sample_weight(
            "balanced",
            y_train,
        )

    fit_start = time.perf_counter()
    fit_checked(
        pipeline,
        predictors.loc[model_training_ids],
        y_train,
        context=f"{sensitivity_id} outer refit seed {winner.split_seed}, {target_name}",
        **fit_parameters,
    )
    test_frame = predictors.loc[test_ids]
    predictions = np.asarray(pipeline.predict(test_frame)).astype(int).reshape(-1)
    scores, score_source = aligned_score_matrix(pipeline, test_frame, labels)
    fitted_transformer = pipeline.named_steps["candidate"]
    return {
        "predictions": predictions,
        "scores": scores,
        "score_source": score_source,
        "selected_columns": fitted_transformer.selected_columns_,
        "selected_groups": fitted_transformer.selected_groups_,
        "engineering_status": fitted_transformer.engineering_status_,
        "training_patients": len(model_training_ids),
        "fit_seconds": time.perf_counter() - fit_start,
    }

## 14. Define one checkpoint unit

One unit is one branch and one split seed. It refits the three frozen winners, creates direct and hard-routed two-stage predictions for the same 312 test patients, and records the refitted configuration.

In [14]:
def evaluate_unit(sensitivity_id, split_seed):
    spec = branch_specs[sensitivity_id]
    training_ids = sorted([*primary_train_ids[split_seed], *spec["extra_training_ids"]])
    test_ids = primary_test_ids[split_seed]
    assert len(test_ids) == 312
    assert set(training_ids).isdisjoint(test_ids)
    assert set(training_ids) | set(test_ids) == set(spec["predictors"].index)

    chosen = {
        target: winners.loc[
            winners["split_seed"].eq(split_seed)
            & winners["target"].eq(target)
        ].iloc[0]
        for target in targets
    }
    random_seed = 500_000 + int(split_seed)
    fitted = {
        target: fit_locked_pipeline(
            sensitivity_id,
            chosen[target],
            training_ids,
            test_ids,
            random_seed,
        )
        for target in targets
    }

    stage_1_prediction = fitted["stage_1"]["predictions"]
    stage_1_scores = fitted["stage_1"]["scores"]
    stage_2_prediction = fitted["stage_2"]["predictions"]
    stage_2_scores = fitted["stage_2"]["scores"]
    two_stage_prediction = np.where(
        stage_1_prediction == 0,
        0,
        stage_2_prediction + 1,
    )
    two_stage_scores = np.column_stack([
        stage_1_scores[:, 0],
        stage_1_scores[:, 1] * stage_2_scores[:, 0],
        stage_1_scores[:, 1] * stage_2_scores[:, 1],
    ])
    assert np.allclose(two_stage_scores.sum(axis=1), 1.0)
    y_test = spec["outcomes"].loc[test_ids].to_numpy()

    new_predictions = []
    for system, predicted, scores in [
        ("direct", fitted["direct"]["predictions"], fitted["direct"]["scores"]),
        ("two_stage", two_stage_prediction, two_stage_scores),
    ]:
        new_predictions.extend({
            "sensitivity_id": sensitivity_id,
            "split_seed": int(split_seed),
            "PATNO": patient,
            "system": system,
            "true_class": int(truth),
            "predicted_class": int(label),
            "score_no_fall": float(score[0]),
            "score_rare_fall": float(score[1]),
            "score_recurrent_fall": float(score[2]),
        } for patient, truth, label, score in zip(test_ids, y_test, predicted, scores))

    new_components = []
    for stage, truth_values, predicted, scores, eligible in [
        ("stage_1", (y_test > 0).astype(int), stage_1_prediction, stage_1_scores, np.ones(len(y_test), dtype=bool)),
        ("stage_2", (y_test == 2).astype(int), stage_2_prediction, stage_2_scores, y_test > 0),
    ]:
        new_components.extend({
            "sensitivity_id": sensitivity_id,
            "split_seed": int(split_seed),
            "PATNO": patient,
            "target": stage,
            "true_class": int(truth),
            "predicted_class": int(label),
            "score_class_0": float(score[0]),
            "score_class_1": float(score[1]),
            "eligible_for_component_metric": bool(is_eligible),
        } for patient, truth, label, score, is_eligible in zip(
            test_ids, truth_values, predicted, scores, eligible,
        ))

    new_selected = []
    for target_name in targets:
        winner = chosen[target_name]
        result = fitted[target_name]
        new_selected.append({
            "sensitivity_id": sensitivity_id,
            "split_seed": int(split_seed),
            "target": target_name,
            "configuration_id": winner.configuration_id,
            "pipeline_key": winner.pipeline_key,
            "candidate_kind": winner.candidate_kind,
            "selector": winner.selector,
            "selector_parameter": winner.selector_parameter,
            "branch": None if pd.isna(winner.branch) else winner.branch,
            "branch_kind": winner.branch_kind,
            "engineering_status": result["engineering_status"],
            "model_family": winner.model_family,
            "grid_id": int(winner.grid_id),
            "parameters": winner.parameters,
            "score_source": result["score_source"],
            "training_patients": result["training_patients"],
            "selected_group_count": len(result["selected_groups"]),
            "selected_groups": " | ".join(sorted(result["selected_groups"])),
            "selected_column_count": len(result["selected_columns"]),
            "selected_column_names": " | ".join(result["selected_columns"]),
            "fit_seconds": result["fit_seconds"],
        })
    return new_predictions, new_components, new_selected

## 15. Lock the run identity

Hash every upstream input and the branch design. Existing checkpoints can resume only when the configuration is unchanged.

In [15]:
RUN_VERSION = "final-notebook-08-sensitivity-analyses-v2-26-features-d28"
REPRODUCTION_SCORE_TOLERANCE = 1e-6


def file_digest(file_path):
    digest = hashlib.sha256()
    with open(file_path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def frame_digest(frame):
    canonical = frame.to_csv(index=False, lineterminator="\n")
    return hashlib.sha256(canonical.encode()).hexdigest()


identity = {
    "run_version": RUN_VERSION,
    "input_sha256": {
        name: file_digest(path)
        for name, path in input_paths.items()
    },
    "branch_manifest_sha256": frame_digest(branch_manifest),
    "added_cohort_patients": added_cohort_ids,
    "refit_seed_rule": "500000 + split_seed",
    "reproduction_score_tolerance": REPRODUCTION_SCORE_TOLERANCE,
    "outer_test_use": "one evaluation per branch using frozen Notebook 07 winners",
}
configuration_hash = hashlib.sha256(
    json.dumps(identity, sort_keys=True).encode()
).hexdigest()
manifest_path = output_directory / "run_manifest.json"
manifest = {
    **identity,
    "configuration_hash": configuration_hash,
    "python": platform.python_version(),
    "scikit_learn": sklearn_version,
    "xgboost": xgboost_version,
    "lightgbm": lightgbm_version,
    "catboost": catboost_version,
}
if manifest_path.is_file():
    existing = json.loads(manifest_path.read_text())
    if existing.get("configuration_hash") != configuration_hash:
        raise RuntimeError(
            "Existing checkpoints belong to a different configuration. "
            "Preserve them and use a new output directory."
        )
else:
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")

print("Configuration hash:", configuration_hash[:16])
print(pd.Series({
    name: manifest[name]
    for name in ["python", "scikit_learn", "xgboost", "lightgbm", "catboost"]
}).to_string())

Configuration hash: 542c2ab52b16d83c
python          3.13.12
scikit_learn      1.8.0
xgboost           3.2.0
lightgbm          4.7.0
catboost         1.2.10


## 16. Prepare resumable checkpoints

Reload rows only for completed branch/seed units. Each completed unit is written atomically, so an interrupted run resumes after the last complete unit.

In [16]:
def atomic_csv(frame, file_path):
    temporary = file_path.with_suffix(file_path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(file_path)


prediction_checkpoint = output_directory / "sensitivity_prediction_checkpoint.csv"
component_checkpoint = output_directory / "sensitivity_component_checkpoint.csv"
selected_checkpoint = output_directory / "sensitivity_selected_pipeline_checkpoint.csv"
completed_path = output_directory / "completed_sensitivity_units.csv"

completed_units = set()
if completed_path.is_file():
    completed_frame = pd.read_csv(completed_path)
    completed_units = set(zip(
        completed_frame["sensitivity_id"],
        completed_frame["split_seed"].astype(int),
    ))


def load_completed_rows(file_path):
    if not file_path.is_file():
        return []
    frame = pd.read_csv(file_path, dtype={"PATNO": "string"}, low_memory=False)
    keep = [
        unit in completed_units
        for unit in zip(frame["sensitivity_id"], frame["split_seed"].astype(int))
    ]
    return frame.loc[keep].to_dict("records")


prediction_rows = load_completed_rows(prediction_checkpoint)
component_rows = load_completed_rows(component_checkpoint)
selected_rows = load_completed_rows(selected_checkpoint)


def run_units(sensitivity_ids):
    planned = [(name, seed) for name in sensitivity_ids for seed in split_seeds]
    remaining = [unit for unit in planned if unit not in completed_units]
    print(
        f"Resuming with {len(planned) - len(remaining)}/{len(planned)} units complete.",
        flush=True,
    )
    session_start = time.perf_counter()
    for position, (sensitivity_id, split_seed) in enumerate(remaining, start=1):
        new_predictions, new_components, new_selected = evaluate_unit(
            sensitivity_id,
            split_seed,
        )
        prediction_rows.extend(new_predictions)
        component_rows.extend(new_components)
        selected_rows.extend(new_selected)
        completed_units.add((sensitivity_id, split_seed))
        atomic_csv(pd.DataFrame(prediction_rows), prediction_checkpoint)
        atomic_csv(pd.DataFrame(component_rows), component_checkpoint)
        atomic_csv(pd.DataFrame(selected_rows), selected_checkpoint)
        atomic_csv(
            pd.DataFrame(sorted(completed_units), columns=["sensitivity_id", "split_seed"]),
            completed_path,
        )
        if split_seed == split_seeds[-1] or position == len(remaining):
            elapsed = time.perf_counter() - session_start
            eta_minutes = (len(remaining) - position) * elapsed / position / 60
            print(
                f"Completed {sensitivity_id} through seed {split_seed:02d}; "
                f"{len(planned) - len(remaining) + position}/{len(planned)} units; "
                f"estimated remaining {eta_minutes:.1f} min",
                flush=True,
            )


print(f"Completed units at start: {len(completed_units)}/{len(branch_specs) * len(split_seeds)}")

Completed units at start: 0/120


# Part C — Primary reproduction gate

## 17. Reproduce the primary outer evaluation

Rerun Notebook 07's 60 outer refits with this notebook's code. This takes under a minute.

In [17]:
run_units(["P00_PRIMARY_REPRODUCTION"])

Resuming with 0/20 units complete.
Completed P00_PRIMARY_REPRODUCTION through seed 19; 20/20 units; estimated remaining 0.0 min


## 18. Confirm exact reproduction before any sensitivity

The reproduced predicted classes and selected columns must match Notebook 07 exactly, and scores must agree within the declared tolerance. This confirms that any sensitivity difference comes from the branch change, not the code. The notebook stops here if the check fails.

In [18]:
score_columns = ["score_no_fall", "score_rare_fall", "score_recurrent_fall"]
reproduced = pd.DataFrame([
    row for row in prediction_rows
    if row["sensitivity_id"] == "P00_PRIMARY_REPRODUCTION"
]).astype({"PATNO": "string"})
reproduction = reproduced.merge(
    primary_predictions.astype({"PATNO": "string"}),
    on=["split_seed", "PATNO", "system"],
    suffixes=("_reproduced", "_primary"),
    validate="one_to_one",
)
reproduced_selected = pd.DataFrame([
    row for row in selected_rows
    if row["sensitivity_id"] == "P00_PRIMARY_REPRODUCTION"
]).merge(
    primary_selected[["split_seed", "target", "selected_column_names"]],
    on=["split_seed", "target"],
    suffixes=("_reproduced", "_primary"),
    validate="one_to_one",
)
max_score_difference = max(
    (reproduction[f"{column}_reproduced"] - reproduction[f"{column}_primary"]).abs().max()
    for column in score_columns
)

reproduction_check = pd.DataFrame([
    {
        "check": "every primary prediction reproduced",
        "passed": len(reproduction) == len(primary_predictions) == 20 * 312 * 2,
        "detail": f"{len(reproduction):,} paired rows",
    },
    {
        "check": "true classes identical",
        "passed": reproduction["true_class_reproduced"].eq(reproduction["true_class_primary"]).all(),
        "detail": "same outcomes and test patients",
    },
    {
        "check": "predicted classes identical",
        "passed": reproduction["predicted_class_reproduced"].eq(reproduction["predicted_class_primary"]).all(),
        "detail": "direct and hard-routed two-stage",
    },
    {
        "check": "scores agree within tolerance",
        "passed": bool(max_score_difference <= REPRODUCTION_SCORE_TOLERANCE),
        "detail": f"maximum absolute difference {max_score_difference:.2e}",
    },
    {
        "check": "refitted selected columns identical",
        "passed": len(reproduced_selected) == 60 and reproduced_selected[
            "selected_column_names_reproduced"
        ].eq(reproduced_selected["selected_column_names_primary"]).all(),
        "detail": "60 seed/target pipelines",
    },
])
display(reproduction_check)
assert reproduction_check["passed"].all(), reproduction_check.loc[~reproduction_check["passed"]]
print("Notebook 07 reproduced exactly; sensitivity branches may now run.")

,check,passed,detail
0,every primary prediction reproduced,True,"12,480 paired rows"
1,true classes identical,True,same outcomes and test patients
2,predicted classes identical,True,direct and hard-routed two-stage
3,scores agree within tolerance,True,maximum absolute difference 5.55e-16
4,refitted selected columns identical,True,60 seed/target pipelines


Notebook 07 reproduced exactly; sensitivity branches may now run.


# Part D — One-time sensitivity evaluation

## 19. Run or resume the five sensitivity branches

Each branch refits the same 60 frozen winners on its changed input and scores the same 312 test patients per seed. This takes about two to three minutes. Progress is shown after each branch.

In [19]:
run_units(SENSITIVITY_IDS)

all_predictions = pd.DataFrame(prediction_rows).astype({"PATNO": "string"}).sort_values(
    ["sensitivity_id", "split_seed", "system", "PATNO"]
).reset_index(drop=True)
all_components = pd.DataFrame(component_rows).astype({"PATNO": "string"}).sort_values(
    ["sensitivity_id", "split_seed", "target", "PATNO"]
).reset_index(drop=True)
all_selected = pd.DataFrame(selected_rows).sort_values(
    ["sensitivity_id", "split_seed", "target"]
).reset_index(drop=True)

is_sensitivity = all_predictions["sensitivity_id"].isin(SENSITIVITY_IDS)
sensitivity_predictions = all_predictions.loc[is_sensitivity].reset_index(drop=True)
sensitivity_components = all_components.loc[
    all_components["sensitivity_id"].isin(SENSITIVITY_IDS)
].reset_index(drop=True)
sensitivity_selected = all_selected.loc[
    all_selected["sensitivity_id"].isin(SENSITIVITY_IDS)
].reset_index(drop=True)
print(f"Sensitivity evaluation complete: {len(sensitivity_predictions):,} predictions.")

Resuming with 0/100 units complete.
Completed S01_101_FLAGS through seed 19; 20/100 units; estimated remaining 1.3 min
Completed S02_ON_OFF through seed 19; 40/100 units; estimated remaining 1.1 min
Completed S03_COHORT_1043 through seed 19; 60/100 units; estimated remaining 0.7 min
Completed S04_REVISION_PREPROCESSING through seed 19; 80/100 units; estimated remaining 0.4 min
Completed S05_CONSTIPATION_LATEST through seed 19; 100/100 units; estimated remaining 0.0 min
Sensitivity evaluation complete: 62,400 predictions.


## 20. Calculate seed-level performance

Report macro F1, all three recalls, balanced accuracy, accuracy, and macro one-vs-rest AUC for every branch, system, and seed. Component metrics describe Stage 1 and Stage 2 separately.

In [20]:
metric_rows = []
confusion_rows = []
for (sensitivity_id, split_seed, system), group in all_predictions.groupby(
    ["sensitivity_id", "split_seed", "system"],
    sort=True,
):
    truth = group["true_class"].to_numpy()
    predicted = group["predicted_class"].to_numpy()
    metric_rows.append({
        "sensitivity_id": sensitivity_id,
        "split_seed": int(split_seed),
        "system": system,
        **multiclass_metrics(truth, predicted, group[score_columns].to_numpy()),
    })
    matrix = confusion_matrix(truth, predicted, labels=[0, 1, 2])
    for true_class in range(3):
        for predicted_class in range(3):
            confusion_rows.append({
                "sensitivity_id": sensitivity_id,
                "split_seed": int(split_seed),
                "system": system,
                "true_class": true_class,
                "predicted_class": predicted_class,
                "patients": int(matrix[true_class, predicted_class]),
            })
all_metrics = pd.DataFrame(metric_rows)
sensitivity_metrics = all_metrics.loc[
    all_metrics["sensitivity_id"].isin(SENSITIVITY_IDS)
].reset_index(drop=True)
sensitivity_confusions = pd.DataFrame(confusion_rows).query(
    "sensitivity_id in @SENSITIVITY_IDS"
).reset_index(drop=True)

component_metric_rows = []
for (sensitivity_id, split_seed, target_name), group in sensitivity_components.groupby(
    ["sensitivity_id", "split_seed", "target"],
    sort=True,
):
    if target_name == "stage_2":
        group = group.loc[group["eligible_for_component_metric"].astype(bool)]
    truth = group["true_class"].to_numpy()
    predicted = group["predicted_class"].to_numpy()
    recalls = recall_score(truth, predicted, labels=[0, 1], average=None, zero_division=0)
    component_metric_rows.append({
        "sensitivity_id": sensitivity_id,
        "split_seed": int(split_seed),
        "target": target_name,
        "patients": len(group),
        "macro_f1": f1_score(truth, predicted, labels=[0, 1], average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(truth, predicted),
        "accuracy": accuracy_score(truth, predicted),
        "recall_class_0": recalls[0],
        "recall_class_1": recalls[1],
        "priority_recall": recalls[0] if target_name == "stage_2" else recalls[1],
    })
sensitivity_component_metrics = pd.DataFrame(component_metric_rows)
print("Seed-level metric rows:", len(sensitivity_metrics))

Seed-level metric rows: 200


## 21. Summarize each branch against the primary run

Average the 20 seed-level estimates for each branch and system, with the Notebook 07 primary result shown for reference. Branch-minus-primary differences are paired by seed and test patients. The corrected interval uses each branch's test/train ratio and remains exploratory because repeated holdouts overlap.

In [21]:
summary_metrics = [
    "macro_f1", "weighted_f1", "accuracy", "balanced_accuracy", "macro_auc_ovr",
    "recall_no_fall", "recall_rare_fall", "recall_recurrent_fall",
]
comparison_metrics = [
    "macro_f1", "balanced_accuracy", "accuracy", "macro_auc_ovr",
    "recall_no_fall", "recall_rare_fall", "recall_recurrent_fall",
]
primary_reference = primary_seed_metrics.assign(sensitivity_id="PRIMARY_NOTEBOOK_07")
system_summary = pd.concat([primary_reference, sensitivity_metrics]).groupby(
    ["sensitivity_id", "system"],
    as_index=False,
    sort=False,
).agg(
    **{f"mean_{metric}": (metric, "mean") for metric in summary_metrics},
    **{f"sd_{metric}": (metric, "std") for metric in summary_metrics},
)


def corrected_interval(differences, test_train_ratio):
    differences = np.asarray(differences, dtype=float)
    mean_difference = float(differences.mean())
    standard_error = float(
        differences.std(ddof=1) * np.sqrt(1 / len(differences) + test_train_ratio)
    )
    if standard_error > 0:
        low, high = stats.t.interval(
            0.95, len(differences) - 1, loc=mean_difference, scale=standard_error,
        )
        p_value = float(2 * stats.t.sf(
            abs(mean_difference / standard_error), df=len(differences) - 1,
        ))
    else:
        low, high, p_value = np.nan, np.nan, np.nan
    return {
        "mean_difference": mean_difference,
        "ci95_low": low,
        "ci95_high": high,
        "corrected_resampled_p_value": p_value,
        "seeds_higher": int((differences > 0).sum()),
        "seeds_lower": int((differences < 0).sum()),
        "n_outer_seeds": len(differences),
        "test_train_ratio": test_train_ratio,
    }


training_size = sensitivity_selected.loc[
    sensitivity_selected["target"].eq("direct")
].groupby("sensitivity_id")["training_patients"].mean()

paired_to_primary = sensitivity_metrics.merge(
    primary_seed_metrics,
    on=["split_seed", "system"],
    suffixes=("", "_primary"),
    validate="many_to_one",
)
difference_columns = ["sensitivity_id", "split_seed", "system"]
for metric in comparison_metrics:
    paired_to_primary[f"sensitivity_minus_primary_{metric}"] = (
        paired_to_primary[metric] - paired_to_primary[f"{metric}_primary"]
    )
    difference_columns.append(f"sensitivity_minus_primary_{metric}")
seed_differences = paired_to_primary[difference_columns]

difference_uncertainty = pd.DataFrame([
    {
        "sensitivity_id": sensitivity_id,
        "system": system,
        "metric": metric,
        **corrected_interval(
            group[f"sensitivity_minus_primary_{metric}"],
            312 / training_size.loc[sensitivity_id],
        ),
    }
    for (sensitivity_id, system), group in seed_differences.groupby(
        ["sensitivity_id", "system"], sort=True,
    )
    for metric in comparison_metrics
])

architecture_rows = []
for sensitivity_id, group in sensitivity_metrics.groupby("sensitivity_id", sort=True):
    paired = group.pivot(index="split_seed", columns="system", values=comparison_metrics)
    for metric in comparison_metrics:
        architecture_rows.append({
            "sensitivity_id": sensitivity_id,
            "metric": metric,
            **corrected_interval(
                paired[(metric, "direct")] - paired[(metric, "two_stage")],
                312 / training_size.loc[sensitivity_id],
            ),
        })
architecture_uncertainty = pd.DataFrame(architecture_rows)

display(system_summary.pivot(
    index="sensitivity_id",
    columns="system",
    values=["mean_macro_f1", "mean_recall_no_fall", "mean_recall_rare_fall", "mean_recall_recurrent_fall"],
).round(3))
display(difference_uncertainty.loc[
    difference_uncertainty["metric"].isin(["macro_f1", "recall_rare_fall"]),
    ["sensitivity_id", "system", "metric", "mean_difference", "ci95_low", "ci95_high", "seeds_higher", "seeds_lower"],
].round(4))

mean_macro_f1           mean_recall_no_fall  \
system                            direct two_stage              direct   
sensitivity_id                                                           
PRIMARY_NOTEBOOK_07                0.513     0.510               0.710   
S01_101_FLAGS                      0.514     0.508               0.712   
S02_ON_OFF                         0.506     0.503               0.713   
S03_COHORT_1043                    0.518     0.510               0.711   
S04_REVISION_PREPROCESSING         0.522     0.517               0.715   
S05_CONSTIPATION_LATEST            0.509     0.509               0.713   

                                     mean_recall_rare_fall            \
system                     two_stage                direct two_stage   
sensitivity_id                                                         
PRIMARY_NOTEBOOK_07            0.743                 0.384     0.351   
S01_101_FLAGS                  0.740                 0.385     0.357   
S02_ON_OFF                     0.741                 0.370     0.343   
S03_COHORT_1043                0.742                 0.388     0.357   
S04_REVISION_PREPROCESSING     0.750                 0.390     0.362   
S05_CONSTIPATION_LATEST        0.741                 0.367     0.351   

                           mean_recall_recurrent_fall            
system                                         direct two_stage  
sensitivity_id                                                   
PRIMARY_NOTEBOOK_07                             0.557     0.522  
S01_101_FLAGS                                   0.548     0.513  
S02_ON_OFF                                      0.540     0.512  
S03_COHORT_1043                                 0.567     0.517  
S04_REVISION_PREPROCESSING                      0.572     0.530  
S05_CONSTIPATION_LATEST                         0.555     0.525

,sensitivity_id,system,metric,mean_difference,ci95_low,ci95_high,seeds_higher,seeds_lower
0,S01_101_FLAGS,direct,macro_f1,0.0007,-0.0172,0.0186,9,11
5,S01_101_FLAGS,direct,recall_rare_fall,0.0007,-0.0493,0.0508,11,7
7,S01_101_FLAGS,two_stage,macro_f1,-0.0016,-0.0296,0.0264,10,10
12,S01_101_FLAGS,two_stage,recall_rare_fall,0.0059,-0.0533,0.0651,12,8
14,S02_ON_OFF,direct,macro_f1,-0.0072,-0.0393,0.0249,4,16
19,S02_ON_OFF,direct,recall_rare_fall,-0.0140,-0.0825,0.0546,7,11
21,S02_ON_OFF,two_stage,macro_f1,-0.0072,-0.0323,0.0178,6,14
26,S02_ON_OFF,two_stage,recall_rare_fall,-0.0081,-0.0600,0.0438,9,10
28,S03_COHORT_1043,direct,macro_f1,0.0052,-0.0250,0.0354,11,9
33,S03_COHORT_1043,direct,recall_rare_fall,0.0037,-0.0388,0.0461,11,4


## 22. Summarize refitted features and branch-specific columns

Count how often each transformed column was retained across seeds. The branch-specific table reports how often the changed columns were selected, including the required `101`-flag selection frequency. These frequencies describe robustness; they do not define a new feature subset.

In [22]:
selected_column_frequency = (
    sensitivity_selected.assign(
        selected_column=sensitivity_selected["selected_column_names"].str.split(" | ", regex=False)
    )
    .explode("selected_column")
    .groupby(["sensitivity_id", "target", "selected_column"], as_index=False)
    .size()
    .rename(columns={"size": "selected_seeds"})
    .sort_values(["sensitivity_id", "target", "selected_seeds"], ascending=[True, True, False])
)

branch_column_rows = []
for sensitivity_id, columns in SENSITIVITY_COLUMNS.items():
    branch_rows = sensitivity_selected.loc[sensitivity_selected["sensitivity_id"].eq(sensitivity_id)]
    for target_name in targets:
        target_rows = branch_rows.loc[branch_rows["target"].eq(target_name)]
        selected_sets = target_rows["selected_column_names"].str.split(" | ", regex=False).map(set)
        for column in columns:
            branch_column_rows.append({
                "sensitivity_id": sensitivity_id,
                "target": target_name,
                "column": column,
                "selected_seeds": int(selected_sets.map(lambda names: column in names).sum()),
                "seeds": len(target_rows),
            })
branch_column_frequency = pd.DataFrame(branch_column_rows)

engineering_status = (
    sensitivity_selected.groupby(["sensitivity_id", "engineering_status"])
    .size()
    .rename("pipelines")
    .reset_index()
)
display(branch_column_frequency.pivot_table(
    index=["sensitivity_id", "column"],
    columns="target",
    values="selected_seeds",
))
display(engineering_status)

target                                                       direct  stage_1  \
sensitivity_id             column                                              
S01_101_FLAGS              NHY_WAS_101_EVER                    13.0     10.0   
                           NP3GAIT_WAS_101_EVER                11.0      9.0   
                           NP3PSTBL_WAS_101_EVER               16.0     13.0   
S02_ON_OFF                 NHY_OFF_MAX                         20.0     20.0   
                           NHY_ON_MAX                          20.0     20.0   
                           NP3GAIT_OFF_MAX                     20.0     20.0   
                           NP3GAIT_ON_MAX                      20.0     20.0   
                           NP3PSTBL_OFF_MAX                    20.0     20.0   
                           NP3PSTBL_ON_MAX                     20.0     20.0   
                           NP3TOT_OFF_MAX                      20.0     20.0   
                           NP3TOT_ON_MAX                       20.0     20.0   
                           OFF_STATE_AVAILABLE                 20.0     20.0   
                           ON_STATE_AVAILABLE                  20.0     20.0   
S03_COHORT_1043            NO_ELIGIBLE_LONGITUDINAL_HISTORY     6.0     14.0   
S04_REVISION_PREPROCESSING DOPTHERST_MISSING                   15.0     19.0   
S05_CONSTIPATION_LATEST    NP1CNST                             11.0     13.0   

target                                                       stage_2  
sensitivity_id             column                                     
S01_101_FLAGS              NHY_WAS_101_EVER                      4.0  
                           NP3GAIT_WAS_101_EVER                  7.0  
                           NP3PSTBL_WAS_101_EVER                 7.0  
S02_ON_OFF                 NHY_OFF_MAX                          19.0  
                           NHY_ON_MAX                           19.0  
                           NP3GAIT_OFF_MAX                      19.0  
                           NP3GAIT_ON_MAX                       19.0  
                           NP3PSTBL_OFF_MAX                     19.0  
                           NP3PSTBL_ON_MAX                      19.0  
                           NP3TOT_OFF_MAX                       19.0  
                           NP3TOT_ON_MAX                        19.0  
                           OFF_STATE_AVAILABLE                  19.0  
                           ON_STATE_AVAILABLE                   19.0  
S03_COHORT_1043            NO_ELIGIBLE_LONGITUDINAL_HISTORY      4.0  
S04_REVISION_PREPROCESSING DOPTHERST_MISSING                     6.0  
S05_CONSTIPATION_LATEST    NP1CNST                               1.0

,sensitivity_id,engineering_status,pipelines
0,S01_101_FLAGS,constructed,15
1,S01_101_FLAGS,not_applicable,45
2,S02_ON_OFF,constructed,8
3,S02_ON_OFF,not_applicable,45
4,S02_ON_OFF,not_constructible,7
5,S03_COHORT_1043,constructed,15
6,S03_COHORT_1043,not_applicable,45
7,S04_REVISION_PREPROCESSING,constructed,15
8,S04_REVISION_PREPROCESSING,not_applicable,45
9,S05_CONSTIPATION_LATEST,constructed,15


## 23. Validate the complete notebook

Confirm exact primary reproduction, complete paired coverage, unchanged frozen configurations, the declared branch rules, valid scores, and bounded metrics.

In [23]:
p00_metrics = all_metrics.loc[all_metrics["sensitivity_id"].eq("P00_PRIMARY_REPRODUCTION")]
p00_vs_primary = p00_metrics.merge(primary_seed_metrics, on=["split_seed", "system"], suffixes=("", "_primary"))
frozen_matches = sensitivity_selected.merge(winners[winner_keys], on=winner_keys, how="inner")
s02_rows = sensitivity_selected.loc[sensitivity_selected["sensitivity_id"].eq("S02_ON_OFF")]
expected_not_constructible = sensitivity_selected["sensitivity_id"].eq("S02_ON_OFF") & (
    sensitivity_selected["branch"].map(
        lambda name: isinstance(name, str)
        and engineering_definitions[name]["kind"] == "interaction"
        and bool(set(engineering_definitions[name]["sources"]) & set(PART_III_COMBINED))
    )
)
engineered = sensitivity_selected["candidate_kind"].eq("engineered_representation")
s03_training = sensitivity_selected.loc[
    sensitivity_selected["sensitivity_id"].eq("S03_COHORT_1043")
    & sensitivity_selected["target"].isin(["direct", "stage_1"]),
    "training_patients",
]
primary_test_pairs = {
    (seed, patient) for seed, patients in primary_test_ids.items() for patient in patients
}

sensitivity_validation = pd.DataFrame([
    {
        "check": "primary reproduction passed before sensitivities",
        "passed": reproduction_check["passed"].all(),
        "detail": f"{len(reproduction_check)} reproduction checks",
    },
    {
        "check": "reproduced seed metrics equal Notebook 07",
        "passed": len(p00_vs_primary) == 40 and all(
            np.allclose(p00_vs_primary[metric], p00_vs_primary[f"{metric}_primary"], atol=1e-12)
            for metric in summary_metrics
        ),
        "detail": "40 seed/system rows",
    },
    {
        "check": "all branch and seed units completed",
        "passed": completed_units == {(name, seed) for name in branch_specs for seed in split_seeds},
        "detail": f"{len(branch_specs)} branches × 20 seeds",
    },
    {
        "check": "each branch and system has 20 seed metrics",
        "passed": sensitivity_metrics.groupby(["sensitivity_id", "system"]).size().eq(20).all()
        and sensitivity_metrics.groupby(["sensitivity_id", "system"]).ngroups == 10,
        "detail": "five branches × two systems",
    },
    {
        "check": "every branch scores the primary test patients once",
        "passed": sensitivity_predictions.groupby(["sensitivity_id", "split_seed", "system"])["PATNO"].nunique().eq(312).all()
        and sensitivity_predictions.groupby(["sensitivity_id", "split_seed", "system", "PATNO"]).size().eq(1).all()
        and set(zip(sensitivity_predictions["split_seed"], sensitivity_predictions["PATNO"])) == primary_test_pairs,
        "detail": "fully paired with Notebook 07",
    },
    {
        "check": "added 1,043-cohort patients train in every seed and are never tested",
        "passed": s03_training.eq(731).all()
        and not sensitivity_predictions["PATNO"].isin(added_cohort_ids).any(),
        "detail": "731 direct and Stage 1 training patients",
    },
    {
        "check": "every refit uses the frozen Notebook 07 configuration",
        "passed": len(frozen_matches) == len(sensitivity_selected) == 5 * 60,
        "detail": "same winner, family, grid row, and parameters",
    },
    {
        "check": "only combined-state interactions are not constructible in S02",
        "passed": sensitivity_selected["engineering_status"].eq("not_constructible").eq(
            expected_not_constructible
        ).all()
        and sensitivity_selected.loc[
            engineered & ~expected_not_constructible, "engineering_status"
        ].eq("constructed").all()
        and int(expected_not_constructible.sum()) == int(
            winners["branch"].map(
                lambda name: isinstance(name, str)
                and engineering_definitions[name]["kind"] == "interaction"
                and bool(set(engineering_definitions[name]["sources"]) & set(PART_III_COMBINED))
            ).sum()
        ),
        "detail": f"{int(expected_not_constructible.sum())} S02 interaction pipelines",
    },
    {
        "check": "S02 never uses combined-state Part III columns",
        "passed": not s02_rows["selected_column_names"].str.contains("COMBINED_MAX", regex=False).any(),
        "detail": "ON/OFF representation only",
    },
    {
        "check": "Stage 2 trains only on true fallers",
        "passed": sensitivity_selected.loc[
            sensitivity_selected["target"].eq("stage_2"), "training_patients"
        ].lt(728).all(),
        "detail": "fewer patients than direct and Stage 1",
    },
    {
        "check": "all refitted pipelines retained columns",
        "passed": sensitivity_selected["selected_column_count"].gt(0).all(),
        "detail": "no empty model input",
    },
    {
        "check": "all scores are finite and sum to one",
        "passed": np.isfinite(sensitivity_predictions[score_columns]).all().all()
        and np.allclose(sensitivity_predictions[score_columns].sum(axis=1), 1.0),
        "detail": "predict_proba or fixed decision-score conversion",
    },
    {
        "check": "sensitivity metrics are finite and bounded",
        "passed": sensitivity_metrics[summary_metrics].apply(
            lambda column: np.isfinite(column).all() and column.between(0, 1).all()
        ).all(),
        "detail": "100 seed/system estimates",
    },
    {
        "check": "run identity still matches",
        "passed": json.loads(manifest_path.read_text())["configuration_hash"] == configuration_hash,
        "detail": "same frozen inputs and branch design",
    },
])
display(sensitivity_validation)
assert sensitivity_validation["passed"].all(), sensitivity_validation.loc[
    ~sensitivity_validation["passed"]
]
print(
    f"Sensitivity validation passed: {sensitivity_validation['passed'].sum()}/"
    f"{len(sensitivity_validation)}"
)

,check,passed,detail
0,primary reproduction passed before sensitivities,True,5 reproduction checks
1,reproduced seed metrics equal Notebook 07,True,40 seed/system rows
2,all branch and seed units completed,True,6 branches × 20 seeds
3,each branch and system has 20 seed metrics,True,five branches × two systems
4,every branch scores the primary test patients ...,True,fully paired with Notebook 07
5,"added 1,043-cohort patients train in every see...",True,731 direct and Stage 1 training patients
6,every refit uses the frozen Notebook 07 config...,True,"same winner, family, grid row, and parameters"
7,only combined-state interactions are not const...,True,7 S02 interaction pipelines
8,S02 never uses combined-state Part III columns,True,ON/OFF representation only
9,Stage 2 trains only on true fallers,True,fewer patients than direct and Stage 1


Sensitivity validation passed: 14/14


## 24. Save the sensitivity artifacts

Save the branch design, audits, patient-level predictions, seed-level metrics, paired comparisons, refitted configurations, and validation tables. Existing differing artifacts are never overwritten.

In [24]:
def frames_equivalent(current, existing):
    if current.columns.tolist() != existing.columns.tolist() or current.shape != existing.shape:
        return False
    for column in current.columns:
        left = current[column]
        right = existing[column]
        left_numeric = pd.to_numeric(left, errors="coerce")
        right_numeric = pd.to_numeric(right, errors="coerce")
        left_numeric_ok = left_numeric.notna().eq(left.notna()).all()
        right_numeric_ok = right_numeric.notna().eq(right.notna()).all()
        if left_numeric_ok and right_numeric_ok:
            if not np.allclose(
                left_numeric.to_numpy(dtype=float),
                right_numeric.to_numpy(dtype=float),
                rtol=1e-12,
                atol=1e-12,
                equal_nan=True,
            ):
                return False
        else:
            left_text = left.astype("string").fillna("<NA>").reset_index(drop=True)
            right_text = right.astype("string").fillna("<NA>").reset_index(drop=True)
            if not left_text.equals(right_text):
                return False
    return True


def save_new_or_equivalent(frame, file_path):
    if file_path.is_file():
        existing = pd.read_csv(file_path, low_memory=False)
        if not frames_equivalent(frame.reset_index(drop=True), existing):
            raise FileExistsError(
                f"Existing artifact truly differs and was not overwritten: {file_path.name}"
            )
        return "already equivalent"
    frame.to_csv(file_path, index=False)
    return "created"


final_artifacts = {
    "sensitivity_branch_manifest.csv": branch_manifest,
    "one_change_audit.csv": one_change_audit,
    "branch_preprocessing_preview.csv": branch_preview,
    "primary_reproduction_check.csv": reproduction_check,
    "sensitivity_predictions.csv": sensitivity_predictions,
    "sensitivity_component_predictions.csv": sensitivity_components,
    "sensitivity_seed_metrics.csv": sensitivity_metrics,
    "sensitivity_component_seed_metrics.csv": sensitivity_component_metrics,
    "sensitivity_confusion_matrices.csv": sensitivity_confusions,
    "sensitivity_system_summary.csv": system_summary,
    "sensitivity_minus_primary_seed_differences.csv": seed_differences,
    "sensitivity_minus_primary_uncertainty.csv": difference_uncertainty,
    "sensitivity_architecture_uncertainty.csv": architecture_uncertainty,
    "sensitivity_selected_pipelines.csv": sensitivity_selected,
    "sensitivity_selected_column_frequency.csv": selected_column_frequency,
    "sensitivity_branch_column_frequency.csv": branch_column_frequency,
    "sensitivity_engineering_status.csv": engineering_status,
    "sensitivity_validation.csv": sensitivity_validation,
}
save_rows = []
for filename, frame in final_artifacts.items():
    status = save_new_or_equivalent(frame, output_directory / filename)
    save_rows.append({"artifact": filename, "status": status, "rows": len(frame)})

display(pd.DataFrame(save_rows))
print(
    "Sensitivity analyses are complete. Their results describe robustness and "
    "cannot revise the primary procedure."
)

,artifact,status,rows
0,sensitivity_branch_manifest.csv,created,6
1,one_change_audit.csv,created,5
2,branch_preprocessing_preview.csv,created,12
3,primary_reproduction_check.csv,created,5
4,sensitivity_predictions.csv,created,62400
5,sensitivity_component_predictions.csv,created,62400
6,sensitivity_seed_metrics.csv,created,200
7,sensitivity_component_seed_metrics.csv,created,200
8,sensitivity_confusion_matrices.csv,created,1800
9,sensitivity_system_summary.csv,created,12


Sensitivity analyses are complete. Their results describe robustness and cannot revise the primary procedure.


## Interpretation boundary

These branches reuse the primary procedure frozen in Notebook 07 and change one assumption at a time. The primary pipeline remains the 1,040-patient, 26-feature analysis with combined-state Part III scores, no `101` flags, current clinical preprocessing, and maximum eligible constipation. Do not replace it with a branch that scores higher here. Because the repeated holdouts overlap and the cohort informed earlier exploration, these are internal robustness checks rather than independent validation.